# 07 — Topology-aware vs FedAvg comparison (Phase 5 evaluation)

Runs the **same protocol twice** per split — once with plain FedAvg, once with the
topology-aware strategy — and compares worst-hospital + mean macro-F1.

**Before running:**
1. Runtime → **GPU** (high-RAM helps; full 1.6M-row cohort).
2. This is **long**: 2 splits x 2 arms x 3 seeds of full runs (~2-4h total).
   Run the **smoke cell first** (minutes) to verify the wiring, then run arms
   selectively. Every arm is one cell, so a failure loses nothing else.
3. Gate (Path B): topology-aware beats FedAvg-best by **>= 0.005** on
   worst-hospital (headline) or mean.


In [ ]:
# 1) Get the code + deps
!git clone -b main https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout main && git pull)
!pip install -q torch_geometric 'flwr[simulation]'

In [ ]:
# 2) Point at the data (mount Drive, set ARMD_DIR before importing amr_fed)
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'   # EDIT to your ARMD folder

In [ ]:
# 3) Sanity: data resolves + imports
import sys
sys.path.insert(0, '/content/Capstone-amr-fed/src')
from amr_fed import config
from pathlib import Path
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong — fix cell 2 and re-run.'
from amr_fed.federated.run import run_fedavg, run_multiseed
from amr_fed.data_loader import load_cohort_frame
from amr_fed.partition import organism_community, specimen_baseline
print('imports OK')

In [ ]:
# 4) Load the cohort ONCE and reuse it for every arm
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.data_loader import load_cohort_frame
df = load_cohort_frame()
print('rows:', f'{len(df):,}')

## Smoke test — single-seed topology run, tiny budget (minutes)

Verifies the wiring end-to-end (fingerprints computed, packed into fit metrics,
strategy mixes by similarity). Numbers are NOT comparable to full runs.

In [ ]:
# 5) SMOKE: specimen split, 1 seed, rounds=2 x local_epochs=2, hidden=64
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_fedavg
from amr_fed.partition import specimen_baseline
smoke = run_fedavg(partition_fn=lambda d, s: specimen_baseline(d), df=df,
                   label='specimen/SMOKE-topology', rounds=2, local_epochs=2,
                   hidden=64, seed=42, strategy='topology',
                   topo_mode='distinctiveness', topo_temperature=1.0)
print('smoke strategy:', smoke['strategy'], '| fedavg_best:', smoke['fedavg_best'])

## Organism-community arms (v2 protocol: 8 rounds x 4 epochs = 32 budget, hidden=128)

In [ ]:
# 6) ORGANISM-COMMUNITY, FedAvg arm (3 seeds)
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import organism_community
org_fed = run_multiseed(seeds=(42, 43, 44), df=df, label='organism-community/FedAvg',
                        partition_fn=lambda d, s: organism_community(d, n_clients=5, seed=s),
                        n_clients=5, rounds=8, local_epochs=4,
                        local_only_epochs=32, hidden=128, strategy='fedavg')

In [ ]:
# 7) ORGANISM-COMMUNITY, topology arm (same protocol, distinctiveness T=1.0)
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import organism_community
org_topo = run_multiseed(seeds=(42, 43, 44), df=df, label='organism-community/TOPO',
                         partition_fn=lambda d, s: organism_community(d, n_clients=5, seed=s),
                         n_clients=5, rounds=8, local_epochs=4,
                         local_only_epochs=32, hidden=128, strategy='topology',
                         topo_mode='distinctiveness', topo_temperature=1.0)

## Specimen arms (notebook-04 cell-7 protocol: 10 rounds x 6 epochs, hidden=128)

In [ ]:
# 8) SPECIMEN, FedAvg arm (3 seeds)
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import specimen_baseline
spec_fed = run_multiseed(seeds=(42, 43, 44), df=df, label='specimen/FedAvg',
                         partition_fn=lambda d, s: specimen_baseline(d),
                         strategy='fedavg')

In [ ]:
# 9) SPECIMEN, topology arm (same protocol, distinctiveness T=1.0)
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import specimen_baseline
spec_topo = run_multiseed(seeds=(42, 43, 44), df=df, label='specimen/TOPO',
                          partition_fn=lambda d, s: specimen_baseline(d),
                          strategy='topology', topo_mode='distinctiveness',
                          topo_temperature=1.0)

## Verdict + report back

Compares topology-aware vs FedAvg per split (Path-B gate: topo wins by >= 0.005 on
worst-hospital or mean). Saves JSON to Drive **and** prints it — paste the block
between the COPY lines back into chat.

In [ ]:
def arm_verdict(fed, topo, name, thresh=0.005):
    def _d(t, f):
        return round(t[0] - f[0], 4) if t[0] is not None and f[0] is not None else None
    d_mean = _d(topo['fedavg_best'], fed['fedavg_best'])
    d_worst = _d(topo['worst_fed'], fed['worst_fed'])
    passed = ((d_worst is not None and d_worst >= thresh)
              or (d_mean is not None and d_mean >= thresh))
    print(f'--- {name}: {"PASS" if passed else "FAIL"} ---')
    print(f'  FedAvg  best {fed["fedavg_best"][0]} +/- {fed["fedavg_best"][1]}  | '
          f'worst {fed["worst_fed"][0]} +/- {fed["worst_fed"][1]}')
    print(f'  Topo    best {topo["fedavg_best"][0]} +/- {topo["fedavg_best"][1]}  | '
          f'worst {topo["worst_fed"][0]} +/- {topo["worst_fed"][1]}')
    print(f'  delta (topo-fedavg): mean {d_mean} | worst {d_worst}  (gate >= {thresh})')
    return {'split': name, 'pass': passed, 'delta_mean': d_mean, 'delta_worst': d_worst}


verdicts = [arm_verdict(org_fed, org_topo, 'organism-community'),
            arm_verdict(spec_fed, spec_topo, 'specimen')]
payload = {'arms': {'organism-community-fedavg': org_fed,
                    'organism-community-topo': org_topo,
                    'specimen-fedavg': spec_fed,
                    'specimen-topo': spec_topo},
           'verdicts': verdicts}
path = '/content/drive/MyDrive/amr_topology_comparison.json'
import json
with open(path, 'w') as f:
    json.dump(payload, f, default=str)
print('saved', path)
print('=== COPY EVERYTHING BELOW THIS LINE BACK TO CHAT ===')
print(json.dumps(payload, default=str))
print('=== END ===')